# 06 — Deep Learning & Clustering (Transformer / LSTM)


In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd()
for candidate in [REPO, REPO.parent, REPO.parent.parent]:
    if (candidate / "src" / "neuro").exists():
        REPO = candidate
        break
sys.path.insert(0, str(REPO / "src"))
import os
os.chdir(REPO)
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
sns.set_theme(style="whitegrid", context="notebook")


In [ ]:

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from neuro.config import PROCESSED_DIR
from neuro.models import build_roi_transformer, build_lstm_classifier, make_tf_dataset
from neuro.mlflow_utils import start_run
from neuro import viz

gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
print("GPUs:", gpus)


In [ ]:

with start_run("06_modeling_dl"):
    X = np.load(PROCESSED_DIR / "roi_ts_stack.npy")
    labels = pd.read_parquet(PROCESSED_DIR / "labels.parquet")
    le = LabelEncoder()
    y = le.fit_transform(labels["group_short"])
    n_time, n_rois = X.shape[1], X.shape[2]

    subj_X, subj_y = [], []
    for sub in labels["subject"].unique():
        idx = labels["subject"] == sub
        subj_X.append(X[idx].mean(axis=0))
        subj_y.append(y[idx][0])
    subj_X = np.stack(subj_X)
    subj_y = np.array(subj_y)

    model = build_roi_transformer(n_rois, n_time, embed_dim=32, num_heads=2, num_layers=2)
    ds = make_tf_dataset(subj_X, subj_y, batch_size=4)
    history = model.fit(ds, epochs=15, validation_split=0.2, verbose=1)
    viz.plot_training_history(history)
    plt.show()
    mlflow.log_metric("final_loss", float(history.history["loss"][-1]))
    mlflow.log_metric("final_accuracy", float(history.history["accuracy"][-1]))


In [ ]:

    conn = np.load(PROCESSED_DIR / "conn_stack.npy")
    kmeans = KMeans(n_clusters=2, random_state=42, n_init=10).fit(conn)
    mlflow.log_metric("kmeans_inertia", float(kmeans.inertia_))

    preds = np.argmax(model.predict(subj_X, verbose=0), axis=1)
    cm = confusion_matrix(subj_y, preds)
    viz.plot_confusion_matrix(cm, list(le.classes_))
    plt.show()
    print(classification_report(subj_y, preds, target_names=le.classes_))

    lstm = build_lstm_classifier(n_rois, n_time)
    lstm_hist = lstm.fit(make_tf_dataset(subj_X, subj_y, batch_size=4), epochs=10, validation_split=0.2, verbose=0)
    viz.plot_training_history(lstm_hist)
    plt.show()
